# Atividade de Revisão — Tratamento de Dados

## Importações

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
!pip install rapidfuzz -q
from rapidfuzz import fuzz
from itertools import combinations
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

## Exercício 1 — Valores Ausentes
Dataset: Melbourne Housing (`melb_data.csv`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/Pastas_Trabalho_de_Ciência_de_Dados/melb_data.csv/melb_data.csv')


In [ ]:
# DIAGNÓSTICO INICIAL
print(df.shape)
df.info()   # mostra tipos e quantidade de não nulos de cada coluna

# RESUMO DE VALORES AUSENTES (QUANTIDADE E %)
resumo = pd.DataFrame({'qtd': df.isnull().sum(), 'pct': (df.isnull().mean() * 100).round(2)})
resumo = resumo[resumo['qtd'] > 0].sort_values('qtd', ascending=False)
print(resumo)

# VISUALIZAÇÃO DO PADRÃO DE AUSÊNCIA
plt.imshow(df.isnull(), aspect='auto', cmap='gray_r')
plt.xticks(range(df.shape[1]), df.columns, rotation=90)
plt.show()
print(df[resumo.index].isnull().corr().round(2))   # perto de 1 = faltam juntas

# ESTRATÉGIA 1 — REMOÇÃO
df_remocao = df.copy()
df_remocao = df_remocao.drop(columns=['BuildingArea', 'YearBuilt']).dropna()

# ESTRATÉGIA 2 — IMPUTAÇÃO
df_imputacao = df.copy()
df_imputacao = df_imputacao.fillna(df.median(numeric_only=True))
df_imputacao['CouncilArea'] = df_imputacao['CouncilArea'].fillna(df['CouncilArea'].mode()[0])

# COMPARAÇÃO ENTRE AS ESTRATÉGIAS
print(pd.DataFrame({'linhas': [len(df), len(df_remocao), len(df_imputacao)],
                    'ausentes': [df.isnull().sum().sum(), df_remocao.isnull().sum().sum(),
                                 df_imputacao.isnull().sum().sum()]},
                   index=['Original', 'Remoção', 'Imputação']))
print(df['BuildingArea'].std(), df_imputacao['BuildingArea'].std())

# RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO
# 1) Sim, as ausências se concentram em 4 das 21 colunas: BuildingArea, YearBuilt,
#    CouncilArea e Car.
# 2) Sim, BuildingArea e YearBuilt costumam faltar nas mesmas linhas
#    (faixas alinhadas na matriz e correlação positiva).
# 3) Remoção: tirei as 2 colunas com 40-47% de ausentes, porque um dropna() direto
#    apagaria mais da metade das linhas; depois removi as poucas linhas restantes.
# 4) Imputação: mediana nas numéricas (resiste a outliers) e moda na CouncilArea
#    (é texto).
# 5) A remoção mantém só dados reais, mas perde duas variáveis. A imputação mantém
#    todas as linhas, mas inventa quase metade dos valores e reduz o desvio padrão.

## Exercício 2 — Duplicatas
Dataset: Customer Transaction Dataset (KPMG), planilha "Transactions"

In [ ]:
arquivo ='/content/drive/MyDrive/Pastas_Trabalho_de_Ciência_de_Dados/KPMG_VI_New_raw_data_update_final.xlsx/KPMG_VI_New_raw_data_update_final.xlsx'
dg = pd.read_excel(arquivo, sheet_name='Transactions')
print(dg.head())

In [ ]:
# CONFERÊNCIA DO CABEÇALHO E DOS TIPOS
print(dg.head())
dg = pd.read_excel(arquivo, sheet_name='Transactions', header=1)
print(dg.head())
dg.info()

# DUPLICATAS EXATAS
print('Duplicatas exatas:', dg.duplicated().sum())

# DUPLICATAS POR SUBCONJUNTO DE COLUNAS
chave = ['customer_id', 'product_id', 'transaction_date', 'list_price']
print('Duplicatas pela chave:', dg.duplicated(subset=chave).sum())
print(dg[dg.duplicated(subset=chave, keep=False)].sort_values(chave))

# UNICIDADE DOS IDENTIFICADORES
print('transaction_id é único?', dg['transaction_id'].is_unique)
print('IDs repetidos:', dg['transaction_id'].duplicated().sum())

# ESTRATÉGIA DE TRATAMENTO E IMPACTO
dg_limpo = dg.drop_duplicates(subset=chave, keep='first')
print('Registros removidos:', len(dg) - len(dg_limpo))
print('Soma list_price antes:', dg['list_price'].sum().round(2))
print('Soma list_price depois:', dg_limpo['list_price'].sum().round(2))

# DESAFIO
marcas = dg['brand'].dropna().unique()
for a, b in combinations(marcas, 2):
    if fuzz.ratio(a, b) > 80:
        print(a, '<->', b, fuzz.ratio(a, b))

demo = pd.read_excel(arquivo, sheet_name='CustomerDemographic', header=1)
print(demo['gender'].value_counts())

# RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO
# 1) Não. O duplicated() só pega linhas 100% iguais. Se a mesma compra foi lançada
#    duas vezes com um ID diferente ou um campo digitado diferente, ela não aparece.
#    Por isso também procurei pela chave (cliente, produto, data e preço).
# 2) Não. Um cliente pode comprar duas bicicletas iguais no mesmo dia, em pedidos
#    separados. As linhas ficam iguais, mas são duas vendas de verdade.
# 3) Porque quem diz o que é "a mesma transação" é a regra da empresa, não o pandas.
#    Se ela vende várias unidades no mesmo pedido, cliente+produto+data não basta.
#    No CRISP-DM isso se resolve no Entendimento do Negócio, conversando com quem
#    conhece o processo antes de mexer nos dados.
# Estratégia: mantive a primeira ocorrência de cada combinação da chave.
#    Foram removidos X registros e a soma de list_price caiu de Y para Z.
# Desafio: na coluna gender aparecem formas diferentes pra mesma coisa
#    (ex.: Female, F e Femal; Male e M), além de U para não informado.

## Exercício 3 — Padronização e Escalonamento
Dataset: Adult Census Income (`adult.data`)

In [ ]:
# NOMES DAS COLUNAS (consulte a documentação do dataset)
colunas = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
           'hours-per-week', 'native-country', 'income']

# CARREGAMENTO DOS DADOS
dh = pd.read_csv('/content/drive/MyDrive/Pastas_Trabalho_de_Ciência_de_Dados/adult/adult.data',
                 header=None, names=colunas)
print(dh.head())
print('Ausentes segundo o pandas:', dh.isna().sum().sum())

# INSPEÇÃO DAS COLUNAS CATEGÓRICAS
cat = dh.select_dtypes('object').columns
for c in cat:
    print(c, dh[c].unique())

# PADRONIZAÇÃO DOS TEXTOS (ANTES x DEPOIS)
antes = dh[cat].nunique()
dh[cat] = dh[cat].apply(lambda s: s.str.strip())
print(pd.DataFrame({'antes': antes, 'depois': dh[cat].nunique()}))
print(dh['workclass'].unique())

# CONVERSÃO DOS MARCADORES DE AUSÊNCIA
dh = dh.replace('?', np.nan)
print(dh.isna().sum())

# ESCALONAMENTO — MIN-MAX
num = dh.select_dtypes('number').columns
dh_minmax = pd.DataFrame(MinMaxScaler().fit_transform(dh[num]), columns=num)

# ESCALONAMENTO — Z-SCORE
dh_zscore = pd.DataFrame(StandardScaler().fit_transform(dh[num]), columns=num)

# ESCALONAMENTO — ROBUSTO
dh_robusto = pd.DataFrame(RobustScaler().fit_transform(dh[num]), columns=num)

# COMPARAÇÃO ENTRE AS TÉCNICAS
for nome, d in [('Min-Max', dh_minmax), ('Z-score', dh_zscore), ('Robusto', dh_robusto)]:
    print(nome)
    print(d.describe().loc[['min', 'max', 'mean', 'std']].round(2))

# RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO
# Inspeção: todos os textos começam com um espaço (ex.: ' State-gov') e o valor
#    desconhecido vem como ' ?'. Não achei diferenças de maiúscula/minúscula nessa base.
# Padronização: tirei os espaços com strip(). A contagem de categorias não mudou,
#    porque todos os valores tinham o mesmo espaço na frente, mas sem isso o
#    replace('?') não funcionaria.
# Ausentes: depois de trocar '?' por NaN apareceram X em workclass, Y em occupation
#    e Z em native-country.
# 1) Porque o pandas só reconhece como ausente o que vem vazio ou como NaN.
#    Aqui o dado faltando foi escrito como '?', que pra ele é um texto qualquer.
# 2) O robusto. Ele usa a mediana e o intervalo entre os quartis, e em capital-gain
#    os dois dão 0, porque quase tudo é zero. Aí ele praticamente não muda a coluna,
#    e o máximo continua perto de 99999. No Min-Max quase tudo fica espremido em 0,
#    e no Z-score os poucos valores altos viram números bem grandes.
# 3) Não. A fnlwgt é um peso do censo: diz quantas pessoas da população aquela
#    linha representa. Não é uma característica da pessoa, então não faz sentido
#    usar como variável num modelo. O certo seria tirar ou usar só como peso.
#    A education-num também é só um código da escolaridade, não uma medida

## Exercício 4 — Outliers
Dataset: Diamonds (seaborn)

In [ ]:
# CARREGAMENTO DOS DADOS
di = sns.load_dataset('diamonds')

# ESTATÍSTICAS DESCRITIVAS E VALORES IMPOSSÍVEIS
print(di.describe().round(2))
print((di[['x', 'y', 'z']] == 0).sum())

# DETECÇÃO — Z-SCORE
num = di.select_dtypes('number').columns
z = (di[num] - di[num].mean()) / di[num].std()
out_z = z.abs() > 3
print(out_z.sum(), 'Linhas com outlier:', out_z.any(axis=1).sum())

# DETECÇÃO — IQR
q1, q3 = di[num].quantile(0.25), di[num].quantile(0.75)
iqr = q3 - q1
out_iqr = (di[num] < q1 - 1.5 * iqr) | (di[num] > q3 + 1.5 * iqr)
print(out_iqr.sum(), 'Linhas com outlier:', out_iqr.any(axis=1).sum())

# VISUALIZAÇÕES
di[num].plot(kind='box', subplots=True, layout=(2, 4), figsize=(14, 6))
plt.show()
sns.scatterplot(data=di, x='carat', y='price', s=5)
plt.show()
sns.scatterplot(data=di, x='x', y='y', s=5)
plt.show()

# ERRO x VALOR EXTREMO LEGÍTIMO
erro = (di[['x', 'y', 'z']] == 0).any(axis=1) | (di[['y', 'z']] > 20).any(axis=1)
print('Erros de registro:', erro.sum())
print(di[di['carat'] > 4])

# TRATAMENTO — REMOÇÃO / CAPPING / TRANSFORMAÇÃO
di = di[~erro]
lim_inf, lim_sup = q1['price'] - 1.5 * iqr['price'], q3['price'] + 1.5 * iqr['price']
di_remocao = di[di['price'].between(lim_inf, lim_sup)]
di_capping = di.copy()
di_capping['price'] = di_capping['price'].clip(lim_inf, lim_sup)
di_log = di.copy()
di_log['price'] = np.log(di_log['price'])

# COMPARAÇÃO ENTRE AS ESTRATÉGIAS
print(pd.DataFrame({'Original': di['price'].describe(), 'Remoção': di_remocao['price'].describe(),
                    'Capping': di_capping['price'].describe(), 'Log': di_log['price'].describe()}).round(2))
fig, ax = plt.subplots(1, 4, figsize=(16, 3))
for a, (nome, d) in zip(ax, [('Original', di), ('Remoção', di_remocao), ('Capping', di_capping), ('Log', di_log)]):
    sns.histplot(d['price'], ax=a)
    a.set_title(nome)
plt.show()

# RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO
# Valores impossíveis: x, y e z são comprimento, largura e profundidade em mm.
#    Tem diamante com medida 0 (8 em x, 7 em y, 20 em z), o que não existe.
#    Também tem y = 58.9 e z = 31.8 mm em pedras de 0.5 a 2 quilates, grande demais.
# Z-score x IQR: o Z-score apontou 2350 linhas e o IQR 6416. O IQR marca bem mais,
#    principalmente em price e depth.
# Critério: considerei erro o que é fisicamente impossível (medida 0 ou acima de
#    20 mm em pedra pequena). Deu 23 linhas, que eu removi. O resto, como diamante
#    grande e caro, é raro mas faz sentido, porque o preço acompanha o tamanho
#    no gráfico carat x price.
# Tratamento: a remoção tirou 3531 linhas e baixou a média de 3931 para 3159.
#    O capping manteve todas as linhas, mas cortou o máximo em 11886.
#    O log não perde nada e deixa a distribuição bem mais simétrica.
# 1) O Z-score supõe que os dados seguem uma distribuição normal (formato de sino).
#    Pra price isso não vale: o histograma Original tem muitos diamantes baratos
#    e uma cauda longa pra direita.
# 2) Depende. Os 5 diamantes acima de 4 quilates custam entre 15 e 18 mil e têm
#    medidas coerentes, então são reais. Se o objetivo é entender o diamante comum,
#    dá pra tirar. Se é precificar pedras grandes, são justamente os mais importantes.
# 3) Uma joalheria querendo saber quais pedras dão mais lucro: as caras e raras
#    seriam removidas como outlier e sumiriam da análise justamente as que mais
#    importam.

## Exercício 5 — Projeto Integrado de Limpeza
Dataset: Dirty E-Commerce Data (Shein), arquivos `us-shein-appliances-3987.csv` e `us-shein-automotive-4110.csv`

In [ ]:
# PERGUNTA DE NEGÓCIO
# Quais categorias de produto têm os maiores descontos, e em que faixa de preço eles
# aparecem?

# CARREGAMENTO DOS ARQUIVOS
import os
pasta = '/content/drive/MyDrive/Pastas_Trabalho_de_Ciência_de_Dados/'
escolhidos = [pasta + 'us-shein-appliances-3987.csv', pasta + 'us-shein-automotive-4110.csv']
de = pd.read_csv(escolhidos[0])

# DIAGNÓSTICO INICIAL (REGISTRAR PROBLEMAS ANTES DE CORRIGIR)
print(de.shape)
de.info()
print(de.head())
print('Duplicatas:', de.duplicated().sum())
print((de.isna().mean() * 100).round(2))

# FUNÇÕES DO PIPELINE
def corrigir_tipos(df):
    df = df.copy()
    for c in df.select_dtypes('object'):
        s = df[c].dropna().astype(str).str.strip()
        if len(s) and s.str.fullmatch(r'\D{0,3}-?[\d.,]+\D{0,3}').mean() > 0.9:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', '')
                                  .str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
    return df

def tratar_ausentes(df):
    df = df.loc[:, df.isna().mean() < 0.5].copy()
    for c in df.select_dtypes('number'):
        df[c] = df[c].fillna(df[c].median())
    return df.fillna('desconhecido')

def tratar_duplicatas(df):
    return df.drop_duplicates()

def padronizar_textos(df):
    df = df.copy()
    for c in df.select_dtypes('object'):
        df[c] = df[c].str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
    return df

def tratar_outliers(df):
    df = df.copy()
    for c in df.select_dtypes('number'):
        q1, q3 = df[c].quantile([0.25, 0.75])
        df[c] = df[c].clip(q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1))
    return df

def criar_features(df, caminho):
    df = df.copy()
    df['categoria'] = os.path.basename(caminho).replace('.csv', '')
    if 'price' in df:
        df['faixa_preco'] = pd.qcut(df['price'], 3, labels=['barato', 'medio', 'caro'], duplicates='drop')
    return df

def limpar(caminho):
    df = pd.read_csv(caminho)
    df = corrigir_tipos(df)
    df = padronizar_textos(df)
    df = tratar_outliers(df)
    df = tratar_ausentes(df)
    df = tratar_duplicatas(df)
    return criar_features(df, caminho)

def qualidade(df):
    return pd.Series({'linhas': len(df), '% ausentes': round(df.isna().mean().mean() * 100, 2),
                      'duplicatas': df.duplicated().sum(),
                      'colunas numéricas': df.select_dtypes('number').shape[1],
                      'colunas de texto': df.select_dtypes('object').shape[1]})

# EXECUÇÃO DO PIPELINE
brutos = {os.path.basename(a): pd.read_csv(a) for a in escolhidos}
limpos = {os.path.basename(a): limpar(a) for a in escolhidos}
de_limpo = pd.concat(limpos.values(), ignore_index=True)
print(de_limpo.head())

# TABELA ANTES x DEPOIS
for nome in brutos:
    print(nome)
    print(pd.DataFrame({'antes': qualidade(brutos[nome]), 'depois': qualidade(limpos[nome])}))
    print(pd.DataFrame({'antes': brutos[nome].dtypes, 'depois': limpos[nome].dtypes}))

# VISUALIZAÇÕES
primeiro = os.path.basename(escolhidos[0])
pd.DataFrame({'antes': de.isna().mean() * 100, 'depois': limpos[primeiro].isna().mean() * 100}).plot(kind='bar')
plt.title('% de ausentes por coluna')
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(x=corrigir_tipos(de)['price'], ax=ax[0]).set_title('price antes')
sns.boxplot(x=limpos[primeiro]['price'], ax=ax[1]).set_title('price depois')
plt.show()

sns.barplot(data=de_limpo, x='categoria', y='discount')
plt.title('Desconto médio por categoria')
plt.show()

# RELATÓRIO E RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO
# Problemas encontrados: preço e desconto vinham como texto ('$12.99', '-27%'),
#    algumas colunas quase vazias, linhas repetidas e títulos com espaço e
#    maiúscula diferentes.
# Decisões:
#    - Tipos: tirei símbolos como $ e % e converti pra número. O desconto ficou
#      positivo (27 em vez de -27) pra facilitar a leitura.
#    - Textos: strip, minúsculo e espaço duplo virando um só, pra que o mesmo
#      produto escrito de jeitos diferentes conte como uma coisa só.
#    - Outliers: capping pelo IQR em vez de remover, pra não perder produto.
#    - Ausentes: tirei colunas com mais de 50% vazio. Nas numéricas usei mediana
#      e no texto coloquei 'desconhecido'.
#    - Duplicatas: removi por último, depois de tudo padronizado.
#    - Features: criei 'categoria' (nome do arquivo) e 'faixa_preco' (barato,
#      médio, caro), que são o que a pergunta de negócio precisa.
#    Resultado: as linhas caíram de X para Y, os ausentes de X% para 0% e as
#    duplicatas de X para 0.
# 1) O mais difícil foi o preço guardado como texto. Olhando o head() parece
#    número normal, e só o info() mostra que a coluna é object. Numa análise
#    rápida, qualquer conta com ela dá erro ou sai errada.
# 2) Importa. Se eu preenchesse os ausentes com a mediana antes de tratar os
#    outliers, ia ter um monte de valores iguais no meio, o IQR ia ficar mais
#    estreito e mais produtos iam ser cortados como outlier. Também deixei as
#    duplicatas pro final, porque textos iguais com espaço ou maiúscula diferentes
#    só viram iguais depois de padronizar.
# 3) Se a pergunta fosse sobre os produtos mais caros ou de luxo, eu não faria
#    capping no preço, porque ele corta justamente esses produtos.